
# Proyecto IA I - Predicción de Calidad del Agua del Río Cauca

## Objetivo
Construir un modelo de clasificación utilizando **Random Forest** para predecir la calidad del agua (`Buena`, `Regular`, `Mala`) a partir de variables fisicoquímicas medidas en estaciones del Río Cauca.

## Modelo final
Se aplicó selección de características basada en la importancia de variables generada por Random Forest, reduciendo el conjunto de atributos de 32 a 15 variables con una pérdida mínima de desempeño.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import joblib


## 1. Carga del dataset

In [ ]:

df = pd.read_csv(
    "../data/raw/Calidad_del_agua_del_Rio_Cauca_20260530.csv"
)

print(df.shape)
df.head()


## 2. Conversión de Oxígeno Disuelto y creación de la variable objetivo

In [ ]:

df["OXIGENO DISUELTO (mg O2/l)"] = pd.to_numeric(
    df["OXIGENO DISUELTO (mg O2/l)"],
    errors="coerce"
)

def clasificar_calidad(oxigeno):

    if pd.isna(oxigeno):
        return np.nan
    elif oxigeno < 3:
        return "Mala"
    elif oxigeno < 5:
        return "Regular"
    else:
        return "Buena"

df["CALIDAD_AGUA"] = df["OXIGENO DISUELTO (mg O2/l)"].apply(
    clasificar_calidad
)

df["CALIDAD_AGUA"].value_counts()


## 3. Eliminación de columnas completamente vacías

In [ ]:

columnas_vacias = [
    col for col in df.columns
    if df[col].isna().sum() == len(df)
]

df = df.drop(columns=columnas_vacias)

print("Columnas eliminadas:", len(columnas_vacias))
print(df.shape)


## 4. Conversión de variables numéricas

In [ ]:

columnas_excluir = [
    "FECHA DE MUESTREO",
    "ESTACIONES",
    "CALIDAD_AGUA"
]

columnas_numericas = [
    col for col in df.columns
    if col not in columnas_excluir
]

for col in columnas_numericas:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df.dtypes.head()


## 5. Construcción del dataset de modelado

In [ ]:

X = df.drop(columns=[
    "CALIDAD_AGUA",
    "OXIGENO DISUELTO (mg O2/l)",
    "FECHA DE MUESTREO",
    "ESTACIONES"
])

y = df["CALIDAD_AGUA"]


In [ ]:

porcentaje_nulos = (X.isnull().sum() / len(X)) * 100

columnas_eliminar = porcentaje_nulos[
    porcentaje_nulos > 70
].index.tolist()

X = X.drop(columns=columnas_eliminar)

print(X.shape)


## 6. Selección de las 15 variables más importantes

In [ ]:

TOP_FEATURES = [
    'CONDUCTIVIDAD ELÉCTRICA (µS/cm)',
    'ALCALINIDAD TOTAL (mg CaCO3/l)',
    'BICARBONATOS (mg CaCO3/l)',
    'DEMANDA BIOQUIMICA DE OXIGENO (mg O2/l)',
    'CLORUROS (mg Cl/l)',
    'DUREZA CALCICA (mg CaCO3/l)',
    'SODIO TOTAL (mg Na/l)',
    'DUREZA TOTAL (mg CaCO3/l)',
    'CALCIO (mg Ca/l)',
    'TURBIEDAD (UNT)',
    'SULFATOS (mg SO4/l)',
    'NITRATOS (mg N-NO3/l)',
    'pH',
    'SOLIDOS SUSPENDIDOS TOTALES (mg SS/l)',
    'SOLIDOS TOTALES (mg SST/l)'
]

X = X[TOP_FEATURES]


In [ ]:

data_model = X.copy()
data_model["TARGET"] = y

data_model = data_model.dropna(subset=["TARGET"])

X = data_model.drop(columns=["TARGET"])
y = data_model["TARGET"]

print(X.shape)
print(y.shape)


## 7. División entrenamiento / prueba

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


## 8. Construcción del Pipeline

In [ ]:

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    (
        "model",
        RandomForestClassifier(
            n_estimators=200,
            random_state=42
        )
    )
])


## 9. Entrenamiento

In [ ]:

pipeline.fit(X_train, y_train)


## 10. Evaluación

In [ ]:

y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))

print(
    "Accuracy:",
    accuracy_score(y_test, y_pred)
)


In [ ]:

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=pipeline.classes_
)

disp.plot()
plt.show()


## 11. Importancia de variables

In [ ]:

importancias = pd.DataFrame({
    "Variable": X.columns,
    "Importancia": pipeline.named_steps["model"].feature_importances_
})

importancias = importancias.sort_values(
    "Importancia",
    ascending=False
)

importancias


## 12. Exportación del modelo

In [ ]:

joblib.dump(
    pipeline,
    "../models/water_quality_pipeline.joblib"
)

print("Modelo exportado correctamente.")
